In [ ]:
from google.colab import drive

# Google Drive を /content/drive にマウント
drive.mount('/content/drive')

%cd /content/drive/MyDrive/research/proj-spectrum_denoising_1d/spectrum_denoising_1d/examples/
!pip install pytorch_lightning

Evaluate the denoiser trained with PE-augmented data.
This version (v2) loads 4D PE data and uses the trained VAE to predict the
underlying signal.

In [ ]:
import sys
import os
import logging

import numpy as np
import torch
from tqdm import tqdm
import pytorch_lightning as pl
import matplotlib.pyplot as plt

sys.path.append('../')
from HDN.models.lvae import LadderVAE
from utils.dataloaders import Dataset

plt.style.use("dark_background")
plt.rc("figure", figsize=[20, 5])
logger = logging.getLogger('pytorch_lightning')
logger.setLevel(logging.WARNING)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Load trained denoiser (PE version).

In [ ]:
denoiser_location = '../dn_checkpoint_pe'
vae = LadderVAE.load_from_checkpoint(
    os.path.join(denoiser_location, 'final_params.ckpt'),
    weights_only=False
).to(device).eval()

Load noisy measurements (PE version) and corresponding ground truth.

In [ ]:
particle_location = f"./../../data/for_1d_denoising/noisy_1d_pe.npy"
particle = np.load(particle_location)
at_particle = particle
print("at_particle.shape", at_particle.shape)
# Expected: (n_samples, 1, channel, 1+d_model) e.g. (100, 1, 200, 5)

gt_particle = np.load("./../../data/for_1d_denoising/gt_1d.npy")
print("gt_particle.shape", gt_particle.shape)

In [ ]:
# Visualise one noisy sample (scalar channel only)
idx = 0
if at_particle.ndim == 4:
    plt.plot(at_particle[idx, 0, :, 0])
else:
    plt.plot(at_particle[idx, 0])
plt.title("Noisy sample (scalar channel)")
plt.show()

Run denoising prediction.

In [ ]:
dataset = Dataset(at_particle)
data_loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)
trainer = pl.Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
)
results = trainer.predict(vae, data_loader)

In [ ]:
denoised = torch.cat(results).cpu().numpy()
print("denoised.shape", denoised.shape)
# Expected: (n_samples, 1, channel)

Compare noisy input, denoised output, and ground truth.

In [ ]:
idx = 0

# Extract scalar channel from noisy input
if at_particle.ndim == 4:
    noisy_signal = at_particle[idx, 0, :, 0]
else:
    noisy_signal = at_particle[idx, 0]

denoised_signal = denoised[idx, 0]
gt_signal = gt_particle[idx, 0]

plt.figure(figsize=(20, 5))
plt.plot(noisy_signal, label='Noisy', alpha=0.5)
plt.plot(denoised_signal, label='Denoised')
plt.plot(gt_signal, label='Ground truth', linestyle='--')
plt.legend()
plt.title('Denoising result (PE version)')
plt.show()